# 05 — Sensitivity Analysis

Notebook 4 gave us a recommendation *under one specific assumption*: that price elasticity is 0.5
(a 10% price increase costs us 5% of purchases). But we made that number up as a reasonable
starting point — the real elasticity is unknown until we've run the test in the real world.

**Question:** would our recommendation change if that assumption were wrong?

We answer this by re-running the full experiment pipeline across a range of plausible elasticity
values and seeing where the decision flips from "Ship" to "Don't ship".

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_processing import BusinessConfig, generate_synthetic_customers
from experiment import TreatmentConfig, run_experiment, summarize_groups
from stats_tools import welch_t_test

plt.style.use("seaborn-v0_8-whitegrid")

customers = generate_synthetic_customers(BusinessConfig(n_customers=20_000, seed=42))


## 1. Sweep elasticity and re-run the analysis pipeline at each value

In [ ]:
elasticity_values = [0.0, 0.2, 0.5, 0.8, 1.0, 1.2, 1.5]
PRICE_INCREASE = 0.10

rows = []
for elasticity in elasticity_values:
    cfg = TreatmentConfig(price_increase_pct=PRICE_INCREASE, elasticity=elasticity, seed=123)
    exp = run_experiment(customers, cfg)

    control_rev = exp.loc[exp["group"] == "control", "revenue"].to_numpy()
    treatment_rev = exp.loc[exp["group"] == "treatment", "revenue"].to_numpy()

    result = welch_t_test(control_rev, treatment_rev, alpha=0.05)

    decision = "Ship" if (result.significant and result.diff > 0) else (
        "Don't ship" if (result.significant and result.diff < 0) else "Borderline / inconclusive"
    )

    rows.append({
        "elasticity": elasticity,
        "implied_purchase_drop_pct": elasticity * PRICE_INCREASE,
        "control_mean": result.control_mean,
        "treatment_mean": result.treatment_mean,
        "revenue_lift_pct": result.diff_relative,
        "p_value": result.p_value,
        "ci_low": result.ci_low,
        "ci_high": result.ci_high,
        "decision": decision,
    })

sensitivity_table = pd.DataFrame(rows)
sensitivity_table


## 2. Where does the decision flip?

Reading the table: as elasticity rises, the demand-destruction effect of the price increase grows,
eating into (and eventually reversing) the revenue gain from higher prices. Somewhere in this range
the recommendation flips from **Ship** to **Don't ship** — that crossover point is the single most
important sensitivity to communicate to the business, because it tells them exactly how wrong our
elasticity assumption would have to be before this decision changes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

colors = sensitivity_table["revenue_lift_pct"].apply(lambda x: "#55A868" if x > 0 else "#C44E52")
ax.bar(sensitivity_table["elasticity"].astype(str), sensitivity_table["revenue_lift_pct"] * 100, color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Assumed price elasticity")
ax.set_ylabel("Revenue lift (%)")
ax.set_title("Revenue impact of a 10% price increase, by elasticity assumption")

for i, row in sensitivity_table.iterrows():
    ax.annotate(row["decision"], (i, row["revenue_lift_pct"] * 100),
                textcoords="offset points", xytext=(0, 8 if row["revenue_lift_pct"] > 0 else -18),
                ha="center", fontsize=8)

plt.tight_layout()
plt.show()


## 3. Confidence intervals across the elasticity sweep

How much does uncertainty (not just the point estimate) change as elasticity increases?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

x = np.arange(len(sensitivity_table))
means = sensitivity_table["treatment_mean"] - sensitivity_table["control_mean"]
lower_err = means - sensitivity_table["ci_low"]
upper_err = sensitivity_table["ci_high"] - means

ax.errorbar(x, means, yerr=[lower_err, upper_err], fmt="o", capsize=6, color="#4C72B0")
ax.axhline(0, color="gray", linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(sensitivity_table["elasticity"])
ax.set_xlabel("Assumed price elasticity")
ax.set_ylabel("Revenue/customer difference ($) with 95% CI")
ax.set_title("Effect size + uncertainty across elasticity assumptions")

plt.tight_layout()
plt.show()


## Takeaways for management

- If the true elasticity is low (demand is fairly price-inelastic), the price increase is a clear win.
- As elasticity rises, the revenue lift shrinks and eventually turns negative — there is a specific
  elasticity threshold (visible in the table/chart above) beyond which we should **not** ship the
  price increase.
- This reframes the decision from a single yes/no into: *"we recommend shipping, and we're
  confident in that as long as elasticity stays below the threshold shown above — here's how we'll
  monitor for that."*

## Next: `06_monte_carlo_simulation.ipynb` — the portfolio differentiator: simulating thousands of
experiments to understand power, error rates, and CI coverage empirically.